# 1. What we are building

**The question:** can a transformer compute an algorithm exactly, with no training?

qkvc takes a short description of an algorithm and places every weight of a transformer that computes it.
These notebooks follow one input through one model, with real values at every step:
the text `Hello Darpan!` through base64.

| Chapter | Shows |
|---|---|
| 1 | base64 manually, and the task file |
| 2 | the token stream, and why fixed distances work for any length |
| 3 | the named numbers of a token |
| 4 | attention as an exact lookup |
| 5 | MLP units as logic gates |
| 6 | the output layer |
| 7 | padding and the end |
| 8 | proving the model correct |
| 9 | other algorithms |
| 10 | the limits |

In [1]:
import base64
import inspect

import helpers
from helpers import TEXT, load

model = load('base64')
written = model.generate('encode', TEXT)
print('model: ', ''.join(written))
print('python:', base64.b64encode(TEXT).decode())

model:  SGVsbG8gRGFycGFuIQ==
python: SGVsbG8gRGFycGFuIQ==


## 1. Base64 manually

Base64 takes 3 bytes, 24 bits, and cuts them into 4 pieces of 6 bits. Each piece picks one of 64 characters.

In [2]:
ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'
group = TEXT[:3]
bits = ''.join(f'{b:08b}' for b in group)
print('bytes:     ', ' '.join(f'{chr(b)!r:8}' for b in group))
print('bits:      ', ' '.join(f'{b:08b}' for b in group))
pieces = [bits[i:i + 6] for i in range(0, 24, 6)]
print('6-bit:     ', ' '.join(pieces))
print('values:    ', ' '.join(str(int(p, 2)) for p in pieces))
print('characters:', ' '.join(ALPHABET[int(p, 2)] for p in pieces))

bytes:      'H'      'e'      'l'     
bits:       01001000 01100101 01101100
6-bit:      010010 000110 010101 101100
values:     18 6 21 44
characters: S G V s


`Hello Darpan!` is 13 bytes: 4 full groups and 1 byte left over.

In [3]:
for i in range(0, len(TEXT), 3):
  chunk = TEXT[i:i + 3]
  print(f'{chunk.decode()!r:7} -> {base64.b64encode(chunk).decode()}')

'Hel'   -> SGVs
'lo '   -> bG8g
'Dar'   -> RGFy
'pan'   -> cGFu
'!'     -> IQ==


## 2. The task file

All the builder knows about base64 is in `tasks/rearrange/base64.py`. The important parts, for encoding:

In [4]:
from tasks.rearrange import base64 as task_file

print(inspect.getsource(task_file.encode_sources))
print(inspect.getsource(task_file.encode_char))

def encode_sources(n):
  """Output character n reads bytes 3 * (n // 4) + (0, 0, 1, 2)[n % 4] and the one after it."""
  first = 3 * (n // 4) + (0, 0, 1, 2)[n % 4]
  return first, first + 1

def encode_char(a, b, place):
  return [a >> 2, ((a & 3) << 4) | (b >> 4), ((a & 15) << 2) | (b >> 6), a & 63][place]


`encode_sources(n)` says which two bytes output `n` reads. `encode_char` says how to make the output from them.
`place` is the output's position in its group of 4.

## 3. What the builder makes from it

In [5]:
r = model.r
print('tokens:              ', len(r.tokens))
print('numbers per token:   ', r.S.size)
print('attention heads:     ', [h.name for h in r.heads])
print('MLP units:           ', len(r.mlp.units))
arrays = [r.table, r.mlp.W_in, r.mlp.b_in, r.mlp.W_out, r.U, r.c] + [w for h in r.heads for w in (h.Q, h.K, h.V, h.O)]
print('parameters:          ', sum(a.size for a in arrays) + r.P * r.S.size)

tokens:               325
numbers per token:    76
attention heads:      ['mode', 'dec_first', 'dec_second', 'enc_first', 'enc_second']
MLP units:            54
parameters:           70603


The model has one attention layer, one MLP layer and an output layer: the parts of any transformer.
Nothing was learned. The next chapters show where each of those numbers comes from.